In [2]:
# ============================================================
#  03b H3 Hex Grid — 统一六边形网格（一次导出多分辨率，互不替代）
#  H3 res 7 ≈ ~1.2 km、res 8 ≈ ~460 m、res 9 ≈ ~174 m（边长量级，随纬度变化）
#  下游约定：
#    - sz_hex_grid_res7.gpkg → 10 OD 采样（粗格，控路由量）
#    - sz_hex_grid_res8.gpkg → 06 / 08 / 09 / 11 分析主格网（与 h3_id 合并）
#    - sz_hex_grid_res9.gpkg → 可选精细分析 / 可视化，保留不删
#  边界：把下载的 shenzhen_boundary.geojson 放在本目录 data_raw/ 下即可运行。
#  运行后会写入 data_out/shenzhen_boundary.geojson（供本目录下游使用）。
#  同仓库里 04–11 步笔记本多数读 ../04 Transport/data_raw/shenzhen_boundary.geojson；
#  请保持与 data_raw/shenzhen_boundary.geojson 一致（复制或软链接即可）。
#  六边形数量会随 h3 小版本略有差异，属正常现象。
# ============================================================

from pathlib import Path
import warnings

import geopandas as gpd
import h3
from shapely.geometry import Polygon, mapping

warnings.filterwarnings("ignore")

OUT = Path("data_out")
OUT.mkdir(exist_ok=True)


def resolve_boundary_path() -> Path:
    """按顺序查找深圳边界：data_raw/geojson → 上次运行生成的 data_out → shp 备选。"""
    candidates = [
        Path("data_raw") / "shenzhen_boundary.geojson",
        OUT / "shenzhen_boundary.geojson",
        Path("data_raw") / "shenzhen_boundary.shp",
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(
        "未找到边界文件。请将下载的 shenzhen_boundary.geojson 保存为：\n"
        "  Data/03 Boundary/data_raw/shenzhen_boundary.geojson\n"
        "（若已跑过一次，也可使用本目录下 data_out/shenzhen_boundary.geojson。）\n"
        "可选：data_raw/shenzhen_boundary.shp（完整 shapefile 套件）。"
    )


def _show_path(p: Path) -> str:
    """在笔记本里优先显示相对路径，便于阅读与协作。"""
    rp = p.resolve()
    try:
        return str(rp.relative_to(Path.cwd().resolve()))
    except ValueError:
        return str(rp)


BOUNDARY = resolve_boundary_path()
print(f"边界来源: {_show_path(BOUNDARY)}")

shenzhen = gpd.read_file(BOUNDARY).to_crs(4326)
shenzhen_geom = shenzhen.union_all() if hasattr(shenzhen, "union_all") else shenzhen.unary_union
poly_geojson = mapping(shenzhen_geom)

boundary_out = OUT / "shenzhen_boundary.geojson"
if BOUNDARY.resolve() != boundary_out.resolve():
    gpd.GeoDataFrame(geometry=[shenzhen_geom], crs="EPSG:4326").to_file(
        boundary_out, driver="GeoJSON"
    )
    print(f"已同步至: {_show_path(boundary_out)}")


def h3_to_polygon(h):
    coords = h3.cell_to_boundary(h)
    return Polygon([(lng, lat) for lat, lng in coords])


for res, label in [(7, "~1.2km"), (8, "~460m"), (9, "~174m")]:
    hexes = h3.geo_to_cells(poly_geojson, res=res)
    hex_gdf = gpd.GeoDataFrame(
        {"h3_id": list(hexes)},
        geometry=[h3_to_polygon(h) for h in hexes],
        crs=4326,
    )
    outfile = OUT / f"sz_hex_grid_res{res}.gpkg"
    hex_gdf.to_file(outfile, driver="GPKG")
    print(f"H3 res {res} ({label}): {len(hex_gdf):,} hexagons -> {outfile}")

print("\nDone.")

边界来源: data_raw/shenzhen_boundary.geojson
已同步至: data_out/shenzhen_boundary.geojson
H3 res 7 (~1.2km): 399 hexagons -> data_out/sz_hex_grid_res7.gpkg
H3 res 8 (~460m): 2,754 hexagons -> data_out/sz_hex_grid_res8.gpkg
H3 res 9 (~174m): 19,253 hexagons -> data_out/sz_hex_grid_res9.gpkg

Done.
